# Grand Mock Contest — Year-2 Capstone Solutions

Reference solutions for all eight problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Problem 1

**Notice:** build a 1D prefix-sum array, then answer each range query as `pre[r] - pre[l-1]`, joined by single spaces.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
idx = 0
n = int(tokens[idx]); idx = idx + 1
a = []
i = 0
while i < n:
    a.append(int(tokens[idx])); idx = idx + 1
    i = i + 1
pre = [0]
i = 0
while i < n:
    pre.append(pre[i] + a[i])
    i = i + 1
q = int(tokens[idx]); idx = idx + 1
out = ""
i = 0
while i < q:
    l = int(tokens[idx]); idx = idx + 1
    r = int(tokens[idx]); idx = idx + 1
    val = pre[r] - pre[l - 1]
    if len(out) > 0:
        out = out + " "
    out = out + str(val)
    i = i + 1
print(out)


## Problem 2

**Notice:** simulate the U/D/L/R moves, staying put whenever the next cell is off the grid, and print the final row and column.

In [ ]:
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
rows = int(header[0]); cols = int(header[1])
start = lines[1].split()
r = int(start[0]); c = int(start[1])
moves = ""
if len(lines) > 2:
    moves = lines[2]
i = 0
while i < len(moves):
    m = moves[i]
    nr = r; nc = c
    if m == "U":
        nr = r - 1
    if m == "D":
        nr = r + 1
    if m == "L":
        nc = c - 1
    if m == "R":
        nc = c + 1
    if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
        r = nr; c = nc
    i = i + 1
print(str(r) + " " + str(c))


## Problem 3

**Notice:** binary-search the smallest maximum-group-sum `cap` for which a greedy left-to-right split needs at most `k` groups (feasibility = `groups_needed(cap) <= k`).

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0]); k = int(tokens[1])
w = []
i = 0
while i < n:
    w.append(int(tokens[2 + i]))
    i = i + 1

def groups_needed(cap):
    groups = 1
    current = 0
    j = 0
    while j < n:
        if current + w[j] > cap:
            groups = groups + 1
            current = w[j]
        else:
            current = current + w[j]
        j = j + 1
    return groups

lo = max(w)
hi = sum(w)
while lo < hi:
    mid = (lo + hi) // 2
    if groups_needed(mid) <= k:
        hi = mid
    else:
        lo = mid + 1
print(str(lo))


## Problem 4

**Notice:** sort the records by a named `skill_of` key, then converge `lo`/`hi` pointers inward to find a pair of skills summing to the target; print `YES`/`NO`.

In [ ]:
import sys

def skill_of(record):
    return record[1]

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0]); target = int(tokens[1])
records = []
i = 0
while i < n:
    pid = int(tokens[2 + 2 * i])
    skill = int(tokens[3 + 2 * i])
    records.append((pid, skill))
    i = i + 1
ordered = sorted(records, key=skill_of)
lo = 0
hi = n - 1
answer = "NO"
while lo < hi and answer == "NO":
    total = ordered[lo][1] + ordered[hi][1]
    if total == target:
        answer = "YES"
    elif total < target:
        lo = lo + 1
    else:
        hi = hi - 1
print(answer)


## Problem 5

**Notice:** BFS from `source` over the adjacency-list graph with a `deque` FIFO queue and a `visited` set, printing the fewest hops to `target` (or `-1` if unreachable).

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
n = int(header[0]); m = int(header[1]); source = int(header[2]); target = int(header[3])
adj = {}
node = 1
while node <= n:
    adj[node] = []
    node = node + 1
i = 0
while i < m:
    parts = lines[1 + i].split()
    u = int(parts[0]); v = int(parts[1])
    adj[u].append(v)
    adj[v].append(u)
    i = i + 1
visited = {source}
queue = deque()
queue.append((source, 0))
answer = "-1"
while len(queue) > 0 and answer == "-1":
    item = queue.popleft()
    node = item[0]
    dist = item[1]
    if node == target:
        answer = str(dist)
    else:
        for nb in adj[node]:
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, dist + 1))
print(answer)


## Problem 6

**Notice:** scan the grid and recursively flood-fill each unvisited `#` region with a `visited` set, tracking the largest region size.

In [ ]:
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
rows = int(header[0]); cols = int(header[1])
grid = []
r = 0
while r < rows:
    grid.append(lines[1 + r])
    r = r + 1
visited = set()

def fill(cr, cc):
    visited.add((cr, cc))
    size = 1
    neighbours = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
    for spot in neighbours:
        nr = spot[0]; nc = spot[1]
        if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
            if grid[nr][nc] == "#" and (nr, nc) not in visited:
                size = size + fill(nr, nc)
    return size

best = 0
r = 0
while r < rows:
    c = 0
    while c < cols:
        if grid[r][c] == "#" and (r, c) not in visited:
            region = fill(r, c)
            if region > best:
                best = region
        c = c + 1
    r = r + 1
print(str(best))


## Problem 7

**Notice:** recursive backtracking that seats each unused person `p` at position `seat` when `p != seat`, marking/unmarking `used`, and counts the complete valid seatings.

In [ ]:
import sys

data = sys.stdin.read()
n = int(data.split()[0])
used = []
build = 0
while build <= n:
    used.append(0)
    build = build + 1

def place(seat):
    if seat > n:
        return 1
    total = 0
    p = 1
    while p <= n:
        if used[p] == 0 and p != seat:
            used[p] = 1
            total = total + place(seat + 1)
            used[p] = 0
        p = p + 1
    return total

print(str(place(1)))


## Problem 8

**Notice:** recursive pre-order walk of the parallel-array binary tree (`-1` marks no child), concatenating node labels into the pre-order sequence.

In [ ]:
import sys

data = sys.stdin.read()
lines = data.split("\n")
n = int(lines[0].split()[0])
label = []
left = []
right = []
build = 0
while build <= n:
    label.append(0)
    left.append(-1)
    right.append(-1)
    build = build + 1
i = 1
while i <= n:
    parts = lines[i].split()
    label[i] = int(parts[0])
    left[i] = int(parts[1])
    right[i] = int(parts[2])
    i = i + 1

def walk(node):
    if node == -1:
        return ""
    text = str(label[node])
    left_text = walk(left[node])
    if len(left_text) > 0:
        text = text + " " + left_text
    right_text = walk(right[node])
    if len(right_text) > 0:
        text = text + " " + right_text
    return text

print(walk(1))
